In [1]:
import numpy as np
import pandas as pd
import random
import json
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, fbeta_score, log_loss, confusion_matrix, mean_squared_log_error, mean_squared_error
from sklearn.impute import KNNImputer
from imblearn.under_sampling import NearMiss 
from imblearn.over_sampling import BorderlineSMOTE
from skmultilearn.problem_transform import LabelPowerset
from sklearn.multioutput import MultiOutputClassifier, ClassifierChain
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import multilabel_confusion_matrix
from sklearn.metrics import hamming_loss
from sklearn.metrics import classification_report as classification_report_multiout

import sys
import math
import shap
import xgboost as xgb

sys.path.insert(0, r"E:\Drive\NOA\MBD-Prediction\Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *

import warnings
warnings.filterwarnings("ignore")

NUTS0 = 'GR'
P = 'P1'
p = P.lower()

file1 = r'E:\Drive\NOA\MBD-Prediction\JRC Interpretability\Greece\data\CMacedonia\GR_CMacedonia_WNV_Dataset_P1_2011_2022.csv'
file2 = r'E:\Drive\NOA\MBD-Prediction\JRC Interpretability\Greece\data\Attica\GR_Attica_WNV_Dataset_P1_2011_2022.csv'
file3 = r'E:\Drive\NOA\MBD-Prediction\JRC Interpretability\Greece\data\Thessaly\GR_Thessaly_WNV_Dataset_P1_2011_2022.csv'

dataset1 = read_data(file1)
dataset2 = read_data(file2)
dataset3 = read_data(file3)

dataset1.head()

dataset2.head()

dataset3.head()

dataset = pd.concat([dataset1, dataset2, dataset3])

dataset.reset_index(drop = True, inplace = True)

print(dataset.shape)

(204, 73)


In [2]:
# def log_loss_vector(y_true, y_pred):
#     e = 1e-15
#     y_pred = np.clip(y_pred, e, 1 - e)
#     log_loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
#     return log_loss


# def optimal_threashold_fbeta_vector(y_prob, y_true, beta = 1, clip_factor = 1e-15, round_factor = 2):
#     if y_prob.shape != y_true.shape:
#         raise Exception("Probability and Predictions matricies are not the same shape")

#     threshold_per_month = []
#     column_range = y_prob.shape[1]

#     for col in range(column_range):
#         precision, recall, thresholds = precision_recall_curve(y_true[:,col], y_prob[:,col])
#         precision = np.clip(precision, clip_factor, 1 - clip_factor)
#         recall = np.clip(recall, clip_factor, 1 - clip_factor)
#         fb_score = ((1 + beta**2) * (precision * recall))/((beta**2 * precision) + recall)
#         #fb_score = ((1 + beta**2) * precision * recall) / ((beta**2 * precision) + recall)
#         fb_max_index = np.argmax(fb_score)
#         optimal_threshold_fb = round(thresholds[fb_max_index], ndigits = round_factor)
#         threshold_per_month.append(optimal_threshold_fb)

#     return np.array(threshold_per_month)

In [3]:
dataset.head()

,NUTS3_ID,year,x,y,ndvi_p1,ndwi_p1,ndmi_p1,ndbi_p1,lst_mean_p1,lst_day_mean_p1,lst_night_mean_p1,lst_min_p1,lst_max_p1,prec_mean_p1,prec_acc_p1,mosq_mean_p1,mosq_sum_p1,bio1,bio2,bio3,bio4,bio5,bio6,bio7,bio8,bio9,bio10,bio11,bio12,bio13,bio14,bio15,bio16,bio17,bio18,bio19,lc_prop1,lc_prop2,lc_prop3,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,distance_to_coast_std,distance_to_river_std,slope_mean_1km_std,aspect_mean_200m_std,elevation_mean_1km_std,hillshade_mean_1km_std,fs_area_1km_std,flow_accu_200m_std,males_lt15,males_15t64,males_gt65,males_total,females_lt15,females_15t64,females_gt65,females_total,mio_eur,l_total,unl_total,cases_total,cases_arr
0,EL521,2011,22.215876,40.552658,0.252698,0.164174,0.372403,-0.372403,6.208697,9.993104,2.424290,-5.821636,3.715091,3.286000,301.706854,433.757576,1301.272727,14.583333,18.666667,49.122807,768.952455,34.0,-4.0,38.0,9.166667,23.833333,23.833333,4.833333,1580.0,347.0,21.0,69.297060,585.0,195.0,195.0,488.0,31,36,30,12,12,1,6,7,12508.751326,1619.864045,10.909091,167.274024,471.994054,174.613165,0.000000,12.577415,6127.601421,866.789289,7.655240,68.916213,547.059733,11.787599,0.00000,40.400688,11806.0,44835.0,13866.0,70507.0,11360.0,44584.0,16384.0,72328.0,2016.25,5025.2,5265.4,6,"[0, 0, 2, 2, 2, 0]"
1,EL522,2011,23.139532,40.695359,0.265538,0.022865,0.253202,-0.253202,7.444165,11.375891,3.512438,-4.746063,5.242598,2.104167,193.713711,278.692913,836.078740,16.125000,20.083333,46.705426,865.533204,39.0,-4.0,43.0,10.333333,26.166667,26.500000,5.333333,1341.0,362.0,8.0,88.115950,512.0,157.0,210.0,391.0,31,30,30,10,10,1,6,6,8521.365773,2204.562938,6.102362,170.376578,294.050327,182.310346,0.006741,5.908999,5609.020284,824.307566,4.668086,44.086739,209.575010,6.368472,0.07597,10.388872,87330.0,375115.0,82051.0,544496.0,84172.0,402246.0,107376.0,593794.0,19627.51,9804.6,10058.8,10,"[0, 0, 3, 4, 3, 0]"
2,EL523,2011,22.758634,41.016242,0.193572,0.101583,0.277345,-0.277345,6.740102,10.879505,2.600698,-5.194396,5.133956,2.543045,233.660050,279.853480,839.560440,16.083333,21.500000,46.739130,890.054476,41.0,-5.0,46.0,9.833333,26.166667,27.000000,5.000000,1417.0,379.0,10.0,78.775725,584.0,198.0,223.0,377.0,31,36,30,12,12,3,5,8,10953.820049,1566.657477,6.461538,153.325412,268.365723,177.491290,0.028224,21.371767,4323.473197,1069.109617,4.638744,47.658551,207.195696,11.096824,0.15371,39.002021,5963.0,24603.0,9769.0,40335.0,5680.0,23436.0,11889.0,41005.0,1124.52,5809.2,5920.4,0,"[0, 0, 0, 0, 0, 0]"
3,EL524,2011,22.109199,40.888821,0.252725,0.133880,0.354723,-0.354723,6.226510,10.217141,2.235878,-5.870000,4.123182,3.739724,343.464625,308.363636,925.090909,14.750000,19.166667,47.916667,790.425662,36.0,-4.0,40.0,9.333333,24.500000,24.500000,4.666667,1630.0,348.0,19.0,63.316479,629.0,219.0,219.0,490.0,31,20,30,12,9,4,4,4,17893.740654,1234.209371,10.625000,142.505681,490.492098,175.009793,0.000000,16.377415,8324.265122,733.091339,9.669614,58.636781,412.361568,15.303624,0.00000,46.596750,11159.0,44537.0,14694.0,70390.0,11052.0,43212.0,17690.0,71954.0,1938.34,5025.2,5265.4,8,"[0, 0, 0, 3, 5, 0]"
4,EL525,2011,22.441443,40.269748,0.243297,0.118603,0.330566,-0.330566,7.617001,11.342695,3.891308,-4.498444,5.272222,2.673776,245.693081,484.000000,1452.000000,15.916667,18.333333,45.833333,769.247960,37.0,-3.0,40.0,15.666667,25.500000,25.500000,6.500000,1470.0,325.0,27.0,70.741701,556.0,174.0,174.0,437.0,31,36,30,12,12,1,6,7,11735.919071,971.980212,13.400000,134.587334,430.793162,170.687146,0.000000,21.108785,5383.969555,789.936868,8.881544,48.186142,455.707394,9.701413,0.00000,46.416540,10433.0,40170.0,12062.0,62665.0,10152.0,40273.0,14620.0,65045.0,1725.03,5025.2,5265.4,0,"[0, 0, 0, 0, 0, 0]"


In [4]:
dataset['cases_arr'] = dataset['cases_arr'].apply(lambda x: json.loads(x))
dataset['cases_arr'] = dataset['cases_arr'].apply(lambda x: np.array(x))
#dataset = convert_multiple_cases_array(dataset, target_col='cases_arr')
dataset['cases_total'] = dataset['cases_total'].apply(lambda x: 1 if x > 0 else 0)
print(dataset.shape)


(204, 73)


In [5]:
dataset.head()

,NUTS3_ID,year,x,y,ndvi_p1,ndwi_p1,ndmi_p1,ndbi_p1,lst_mean_p1,lst_day_mean_p1,lst_night_mean_p1,lst_min_p1,lst_max_p1,prec_mean_p1,prec_acc_p1,mosq_mean_p1,mosq_sum_p1,bio1,bio2,bio3,bio4,bio5,bio6,bio7,bio8,bio9,bio10,bio11,bio12,bio13,bio14,bio15,bio16,bio17,bio18,bio19,lc_prop1,lc_prop2,lc_prop3,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,distance_to_coast_std,distance_to_river_std,slope_mean_1km_std,aspect_mean_200m_std,elevation_mean_1km_std,hillshade_mean_1km_std,fs_area_1km_std,flow_accu_200m_std,males_lt15,males_15t64,males_gt65,males_total,females_lt15,females_15t64,females_gt65,females_total,mio_eur,l_total,unl_total,cases_total,cases_arr
0,EL521,2011,22.215876,40.552658,0.252698,0.164174,0.372403,-0.372403,6.208697,9.993104,2.424290,-5.821636,3.715091,3.286000,301.706854,433.757576,1301.272727,14.583333,18.666667,49.122807,768.952455,34.0,-4.0,38.0,9.166667,23.833333,23.833333,4.833333,1580.0,347.0,21.0,69.297060,585.0,195.0,195.0,488.0,31,36,30,12,12,1,6,7,12508.751326,1619.864045,10.909091,167.274024,471.994054,174.613165,0.000000,12.577415,6127.601421,866.789289,7.655240,68.916213,547.059733,11.787599,0.00000,40.400688,11806.0,44835.0,13866.0,70507.0,11360.0,44584.0,16384.0,72328.0,2016.25,5025.2,5265.4,1,"[0, 0, 2, 2, 2, 0]"
1,EL522,2011,23.139532,40.695359,0.265538,0.022865,0.253202,-0.253202,7.444165,11.375891,3.512438,-4.746063,5.242598,2.104167,193.713711,278.692913,836.078740,16.125000,20.083333,46.705426,865.533204,39.0,-4.0,43.0,10.333333,26.166667,26.500000,5.333333,1341.0,362.0,8.0,88.115950,512.0,157.0,210.0,391.0,31,30,30,10,10,1,6,6,8521.365773,2204.562938,6.102362,170.376578,294.050327,182.310346,0.006741,5.908999,5609.020284,824.307566,4.668086,44.086739,209.575010,6.368472,0.07597,10.388872,87330.0,375115.0,82051.0,544496.0,84172.0,402246.0,107376.0,593794.0,19627.51,9804.6,10058.8,1,"[0, 0, 3, 4, 3, 0]"
2,EL523,2011,22.758634,41.016242,0.193572,0.101583,0.277345,-0.277345,6.740102,10.879505,2.600698,-5.194396,5.133956,2.543045,233.660050,279.853480,839.560440,16.083333,21.500000,46.739130,890.054476,41.0,-5.0,46.0,9.833333,26.166667,27.000000,5.000000,1417.0,379.0,10.0,78.775725,584.0,198.0,223.0,377.0,31,36,30,12,12,3,5,8,10953.820049,1566.657477,6.461538,153.325412,268.365723,177.491290,0.028224,21.371767,4323.473197,1069.109617,4.638744,47.658551,207.195696,11.096824,0.15371,39.002021,5963.0,24603.0,9769.0,40335.0,5680.0,23436.0,11889.0,41005.0,1124.52,5809.2,5920.4,0,"[0, 0, 0, 0, 0, 0]"
3,EL524,2011,22.109199,40.888821,0.252725,0.133880,0.354723,-0.354723,6.226510,10.217141,2.235878,-5.870000,4.123182,3.739724,343.464625,308.363636,925.090909,14.750000,19.166667,47.916667,790.425662,36.0,-4.0,40.0,9.333333,24.500000,24.500000,4.666667,1630.0,348.0,19.0,63.316479,629.0,219.0,219.0,490.0,31,20,30,12,9,4,4,4,17893.740654,1234.209371,10.625000,142.505681,490.492098,175.009793,0.000000,16.377415,8324.265122,733.091339,9.669614,58.636781,412.361568,15.303624,0.00000,46.596750,11159.0,44537.0,14694.0,70390.0,11052.0,43212.0,17690.0,71954.0,1938.34,5025.2,5265.4,1,"[0, 0, 0, 3, 5, 0]"
4,EL525,2011,22.441443,40.269748,0.243297,0.118603,0.330566,-0.330566,7.617001,11.342695,3.891308,-4.498444,5.272222,2.673776,245.693081,484.000000,1452.000000,15.916667,18.333333,45.833333,769.247960,37.0,-3.0,40.0,15.666667,25.500000,25.500000,6.500000,1470.0,325.0,27.0,70.741701,556.0,174.0,174.0,437.0,31,36,30,12,12,1,6,7,11735.919071,971.980212,13.400000,134.587334,430.793162,170.687146,0.000000,21.108785,5383.969555,789.936868,8.881544,48.186142,455.707394,9.701413,0.00000,46.416540,10433.0,40170.0,12062.0,62665.0,10152.0,40273.0,14620.0,65045.0,1725.03,5025.2,5265.4,0,"[0, 0, 0, 0, 0, 0]"


In [6]:
X = dataset.drop(columns=['cases_total','cases_arr'])
y = dataset['cases_arr']
s = dataset['cases_total']

In [7]:
n_folds = 5
#k_fold = StratifiedKFold(n_splits = n_folds, shuffle = True, random_state = 0)
k_fold = KFold(n_splits = n_folds, shuffle = True, random_state = 0)

#train_idx, test_idx, _, _ = train_test_split(X.index, y, stratify = s, test_size = 0.3, random_state = 0)
#train_idx, test_idx, _, _ = train_test_split(X.index, y, test_size = 0.25, random_state = 0)

# year_out = [2017, 2018]

# train_idx = X[~X['year'].isin(year_out)].index
# test_idx = X[X['year'].isin(year_out)].index

In [8]:
# print(f"Training set 0/1 Ratio: {(dataset.iloc[train_idx]['cases_total'].value_counts().get(0) / dataset.iloc[train_idx]['cases_total'].value_counts().get(1)):.2f}")
# print(f"Test set 0/1 Ratio: {(dataset.iloc[test_idx]['cases_total'].value_counts().get(0) / dataset.iloc[test_idx]['cases_total'].value_counts().get(1)):.2f}")

In [9]:
features_to_remove = ['x', 'y', 'year']
feature_names = X.select_dtypes(exclude=['object']).drop(columns = features_to_remove).columns
print(feature_names)

Index(['ndvi_p1', 'ndwi_p1', 'ndmi_p1', 'ndbi_p1', 'lst_mean_p1',
       'lst_day_mean_p1', 'lst_night_mean_p1', 'lst_min_p1', 'lst_max_p1',
       'prec_mean_p1', 'prec_acc_p1', 'mosq_mean_p1', 'mosq_sum_p1', 'bio1',
       'bio2', 'bio3', 'bio4', 'bio5', 'bio6', 'bio7', 'bio8', 'bio9', 'bio10',
       'bio11', 'bio12', 'bio13', 'bio14', 'bio15', 'bio16', 'bio17', 'bio18',
       'bio19', 'lc_prop1', 'lc_prop2', 'lc_prop3', 'lc_type1', 'lc_type2',
       'lc_type3', 'lc_type4', 'lc_type5', 'distance_to_coast',
       'distance_to_river', 'slope_mean_1km', 'aspect_mean_200m',
       'elevation_mean_1km', 'hillshade_mean_1km', 'fs_area_1km',
       'flow_accu_200m', 'distance_to_coast_std', 'distance_to_river_std',
       'slope_mean_1km_std', 'aspect_mean_200m_std', 'elevation_mean_1km_std',
       'hillshade_mean_1km_std', 'fs_area_1km_std', 'flow_accu_200m_std',
       'males_lt15', 'males_15t64', 'males_gt65', 'males_total',
       'females_lt15', 'females_15t64', 'females_gt65', 'f

In [10]:
## XGBoost Model Parameters

params = {
    'max_depth': 3,
    'verbosity': 0,
    'seed': 0,
    'tree_method': 'hist',
    'device': 'cuda',
    'objective': 'reg:squaredlogerror',
    'eval_metric': 'rmsle',
}

In [11]:
fold_counter = 0
y_train_list = []
y_test_list = []
y_train_pred_list = []
y_test_pred_list = []
#shap_list = []

for train_idx, test_idx in k_fold.split(X, s):
    print(f'Fold: {fold_counter+1}/{n_folds}')
    X_train = X.iloc[train_idx].select_dtypes(exclude=['object']).drop(columns = features_to_remove)
    X_test = X.iloc[test_idx].select_dtypes(exclude=['object']).drop(columns = features_to_remove)

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    print(f"Training set 0/1 Ratio: {(dataset.iloc[train_idx]['cases_total'].value_counts().get(0) / dataset.iloc[train_idx]['cases_total'].value_counts().get(1)):.2f}")
    print(f"Test set 0/1 Ratio: {(dataset.iloc[test_idx]['cases_total'].value_counts().get(0) / dataset.iloc[test_idx]['cases_total'].value_counts().get(1)):.2f}")
    
    scaler = MinMaxScaler()
    scaler.fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
            
    imputer = KNNImputer()
    imputer.fit(X_train)
    X_train_imputed = imputer.transform(X_train_scaled)
    X_test_imputed = imputer.transform(X_test_scaled)

    y_train_np = np.stack(y_train.to_numpy())
    y_test_np = np.stack(y_test.to_numpy())

    model = xgb.XGBRegressor(**params)   
    trained_model = model.fit(X_train_imputed, y_train_np)

    y_train_pred = trained_model.predict(X_train_imputed)
    y_test_pred = trained_model.predict(X_test_imputed)

    # explainer = shap.KernelExplainer(trained_model.predict_proba, X_train_imputed, link = 'logit')
    # explainer = shap.KernelExplainer(trained_model.predict_proba, shap.kmeans(X_train_imputed, 20), link = 'logit')
    # shap_values = explainer.shap_values(X_train_imputed)

    y_train_list.append(y_train_np)
    y_test_list.append(y_test_np)
    y_train_pred_list.append(y_train_pred)
    y_test_pred_list.append(y_test_pred)
    #shap_list.append(shap_values)
    fold_counter += 1

Fold: 1/5
Training set 0/1 Ratio: 1.33
Test set 0/1 Ratio: 1.93
Fold: 2/5
Training set 0/1 Ratio: 1.36
Test set 0/1 Ratio: 1.73
Fold: 3/5
Training set 0/1 Ratio: 1.59
Test set 0/1 Ratio: 0.95
Fold: 4/5
Training set 0/1 Ratio: 1.47
Test set 0/1 Ratio: 1.28
Fold: 5/5
Training set 0/1 Ratio: 1.41
Test set 0/1 Ratio: 1.50


In [12]:
y_train_true = np.concatenate(y_train_list, axis=0)
y_test_true = np.concatenate(y_test_list, axis=0)

y_train_pred = np.concatenate(y_train_pred_list, axis = 0)
y_test_pred = np.concatenate(y_test_pred_list, axis = 0)

In [13]:
y_train_pred = np.round(y_train_pred, 0).astype(int)
y_test_pred = np.round(y_test_pred, 0).astype(int)

y_train_pred[y_train_pred < 0] = 0
y_test_pred[y_test_pred < 0] = 0

In [14]:
train_rmse = mean_squared_error(y_train_true, y_train_pred)
print(f"Train RMSE: {train_rmse:.4f}")

test_rmse = mean_squared_error(y_test_true, y_test_pred)
print(f"Test RMSE: {test_rmse:.4f}")

Train RMSE: 5.5266
Test RMSE: 9.8374


In [15]:
train_rmsle = mean_squared_log_error(y_train_true, y_train_pred)
print(f"Train RMSLE: {train_rmsle:.4f}")

test_rmsle = mean_squared_log_error(y_test_true, y_test_pred)
print(f"Test RMSLE: {test_rmsle:.4f}")

Train RMSLE: 0.0315
Test RMSLE: 0.3487


In [16]:
y_train_pred[y_train_pred >= 1] = 1
y_test_pred[y_test_pred >= 1] = 1
y_train_true[y_train_true >= 1] = 1
y_test_true[y_test_true >= 1] = 1

In [17]:
hm_loss = hamming_loss(y_test_true, y_test_pred)
print(f"Hamming Loss: {hm_loss}")

Hamming Loss: 0.22794117647058823


In [18]:
print(classification_report_multiout(y_test_true, y_test_pred, output_dict=False, target_names=['May', 'June', 'July', 'August', 'September', 'October']))

              precision    recall  f1-score   support

         May       0.00      0.00      0.00         1
        June       0.14      0.33      0.20         3
        July       0.22      0.42      0.29        43
      August       0.50      0.84      0.63        75
   September       0.46      0.86      0.60        63
     October       0.29      0.28      0.29        25

   micro avg       0.40      0.68      0.51       210
   macro avg       0.27      0.45      0.33       210
weighted avg       0.40      0.68      0.50       210
 samples avg       0.29      0.29      0.27       210



In [19]:
'''
MCM = [[TN, FP],
       [FN, TP]]
'''
labels = [0,1,2,3,4,5]

cm = multilabel_confusion_matrix(y_test_true, y_test_pred, labels=labels)

print(cm)

[[[203   0]
  [  1   0]]

 [[195   6]
  [  2   1]]

 [[ 98  63]
  [ 25  18]]

 [[ 66  63]
  [ 12  63]]

 [[ 78  63]
  [  9  54]]

 [[162  17]
  [ 18   7]]]


In [20]:
shap.summary_plot(shap_values = shap_values, features = feature_names)

NameError: name 'shap_values' is not defined

In [ ]:
unique_values, counts = np.unique(y_test_np.flatten(), return_counts=True)

print("Value counts in y_test")
for value, count in zip(unique_values, counts):
    print(f"{value} : {count}")

In [ ]:
sum0 = 0
sum1 = 0
for matrix in cm:
    sum0 += np.sum(matrix[0])
    sum1 += np.sum(matrix[1])

print(f"Value counts in y_test\n0 : {sum0}\n1 : {sum1}")

In [ ]:
from sklearn.metrics import precision_recall_curve

proba_per_month = []
gt_per_month = []

for col in range(y_train_pred.shape[1]):
    proba_per_month.append(y_train_pred[:,col])
    gt_per_month.append(y_train_np[:,col])

In [ ]:
results_temp = {'score': proba_per_month[0], 'case': gt_per_month[0]}
df_results = pd.DataFrame(results_temp)

plot_pr_curve(df_results, beta= 1, plot_fbeta = True)